# Week 6: Deep Learning for Remote Sensing

In this week's practical, we will take a first look at working with remote sensing data.
We won't dive into deep learning, due to time restrictions.
Instead, we will focus on understanding what satellite data looks like in practice and how to perform basic spatial computations.

## Setup

To start, let's download the dataset for today's session.
This contains images of the Amazon rainforest captured by the Sentinel-2 mission

In [ ]:
%%bash
gdown 1Om7BymjifbiQByalKcG0AkFitgRWe50f
unzip amazon_sentinel2.zip

The data is now ready in the `data` folder.


In [ ]:
from pathlib import Path

data_dir = Path("data")

## Raster Files

In this section, we will work with two images from [Sentinel-2](https://www.esa.int/Applications/Observing_the_Earth/Copernicus/Sentinel-2), a mission consisting of two satellites that record multispectral data (13 bands) at up to 10m resolution.

### What is a raster?

Optical satellite data are raster images, which means they are a rectangular grid of pixels.
You mayb have encountered "JPEG" or "PNG" image files, however, satellite images are often stored as **TIFF** files.
Unlike JPEGs, TIFFs are usually uncompressed and can store more than just the three "RGB" channels.

Every pixel in a satellite image is georeferenced.
This means each pixel coordinates with a specific square on the Earth's surface.
When a TIFF stores this spatial information, we call it a **GeoTIFF**.
This is usually done by storing two key pieces of metadata.
First is the **Coordinate Reference System (CRS)**, which tells the computer which map projection is being used (more on this later).
Second is the **Bounding Box**, which defines the boundaries of the entire image in this coordinate system.

Optical satellite data are _raster images_, which means they are a rectangular grid of pixels.
Satellite images are often stored and shared as "TIFF" files, which is an image format.
Unlike "JPEG", "TIFF" files are not compressed and can also have many channels.
Also every pixel in satellite image is georeferenced, meaning that each pixel corresponds to a square in geographic space.
TIFF images can store information about this georefencing, in which case they become GeoTIFFs.

### Data levels

Remote sensing data is categorised into "levels" based on processing.
**Level 0** is raw data exactly as recorded by the sensors.
**Level 1** is processed to correct for instrument distortions.
**Level 2**, which is what we are using, is corrected for basic atmospheric distortions like clouds and haze.
**Level 3** is a composite where multiple images are combined to remove clouds or create a larger grid.
Finally, **Level 4** represents derived products, such as maps containing modeled data or results of analysis.

Every satellite will define their levels a bit differently.
However, here we will use Level 2A images.

### Loading data

To load the GeoTIFF data, we will use the library `rasterio`.
Unlike standard image libraries like `Pillow` (or `PIL`), `rasterio` natively understands georeferenced data.
This means it can read spatial metadata and perform common geographic transformations.

In [ ]:
import rasterio
import matplotlib.pyplot as plt

# The sentinel 2 data is in this folder
sentinel_data = data_dir / "Sentinel-2"

# Let's take the folder corresponding to one day in 2024
day1_dir = sentinel_data / "2024-06-06"

# Get all files from the day1 folder
day1_files = list(day1_dir.glob("*.tiff"))

# Have a peek at which files were found
day1_files

Notice there are multiple TIFF files in this folder.
These correspond to the different bands that were captured.
Sentinel-2 carries the Multispectral Imager (MSI), which delivers 13 spectral bands with different spatial resolutions.

- The blue (B2), green (B3), red (B4), and near-infrared (B8) channels have a 10-meter resolution.
- The red edge (B5), near-infrared NIR (B6, B7, and B8A), and short-wave infrared SWIR (B11 and B12) have a ground sampling distance of 20 meters.
- The coastal aerosol (B1) and cirrus band (B10) have a 60-meter pixel size.

In [ ]:
# Lets map every band to its path so we can find the image of a given band more easily
day1_channels = {
    path.stem: path
    for path in day1_files
}

# Let's list the bands and sort them numerically
bands = list(sorted(day1_channels.keys()))

Now we can read a single file.
We will take the file corresponding to band B2 (blue) and use `rasterio` to open it.

Before we do, there is one final point to understand about the pixel data.
The intensity values in these Sentinel-2 files are stored as 16-bit integers, meaning they can range from 0 to 65,535.
Standard photos are typically 8-bit (0 to 255), but that is far too coarse for the high-precision measurements needed in satellite imagery.
To make these values easier to work with, we will map them to a 0–1 range, where 1 represents the maximum possible intensity.

In [ ]:
# Take the file corresponding to the band B01
example_file = day1_channels["B02"]

# Max intensity for 16-bit data
max_intensity = 65535

# Use rasterio to open the file
with rasterio.open(example_file, "r") as dataset:
    # First we check the coordinate reference system (CRS)
    crs = dataset.crs

    # Then we check the coordinate bounds that mark the edges of this image
    bounds = dataset.bounds
    print(f"Image bounds: {bounds}")

    # Here we read the image into a numpy array and scale to 0-1 range
    band = dataset.read() / max_intensity

# Let's see what the shape of the resulting read is
print(band.shape)

Notice that the image has three dimensions.
This is because TIFF files can store more than one channel, so the first dimension represents the channels even if there is only one.
To work with the image pixels directly, we will take the first and only channel.

Now let's try plotting the single B01 band

In [ ]:
# Extract the first channel
band_channel = band[0]

# Plot the single B01 band
# We use the 'extent' argument to tell matplotlib the geographic limits of the axes
plt.imshow(band_channel, extent=(bounds.left, bounds.right, bounds.bottom, bounds.top), cmap="gray")
plt.title("Sentinel-2 Band B02")

### Creating a True Color Image

Now we can combine the red (B4), green (B3), and blue (B2) channels into a single 3-channel image.
This allows us to visualise the scene in "True Color", which is how the area would look to the human eye.

To do this, we need to read each band separately and "stack" them together.
Standard plotting libraries like matplotlib expect 3-channel images to have the color dimensions at the very end of the array (height, width, channels), so we will adjust our data to match that format.

In [ ]:
import numpy as np

# Create an empty list to hold our arrays
stack = []

# Loop through the specific bands needed for RGB
for band in ["B04", "B03", "B02"]:

    # Get the path to the TIFF with the band info
    path = day1_channels[band]

    # Read the raster, take the first channel, and scale it
    with rasterio.open(path, "r") as dataset:
        band = dataset.read()[0]
        stack.append(band / max_intensity)

# Stack bands to form a 3D array where the last axis is the channels dimension
# This transforms our data from (3, height, width) to (height, width, 3)
stack = np.stack(stack, axis=-1)

# Plot the images
plt.imshow(stack, extent=(bounds.left, bounds.right, bounds.bottom, bounds.top))
plt.title("Sentinel-2 True Color (RGB)")

The resulting image usually looks a bit dark or dull.
This happens because satellite sensors are designed to be incredibly sensitive.
As a result, the raw values in a typical scene often cluster in the lower end of the 0–1 range.

To make the image clearer for our eyes, we can multiply the intensity values by a factor of 10.

In [ ]:
plt.imshow(stack * 10, extent=(bounds.left, bounds.right, bounds.bottom, bounds.top))
plt.title("Sentinel-2 True Color (RGB)")

## Coordinate Reference Systems

So what is a coordinate reference system?
Well, while we may be used to the idea of points on the earth having a latitude and longitude, in reality this is quite a radical idea.
A coordinate reference system is, simply put, a way of assigning "coordinates" to the points on earth.
There are many ways of doing it, and the usual lat-lon is only one of them (officially called **EPSG:4326**).

![Coordinate Reference System](https://earthdatascience.org/images/courses/earth-analytics/spatial-data/geographic-origin.png)

Because the earth is a round curved object, our maps in coordinate space will always be distorted or warped in some way.
They are only a representation of reality, not reality itself.
This distortion is partly why there exist many different coordinate systems.
Different CRS will distort things differently, and this may inform your choice.

If you are working only on a smaller region of the earth, it is best to pick a coordinate system that minimises distortion on that region, such as UTM.
Other projections might distort distances between points but will at least keep the areas of shapes accurate.

Another reason to change your coordinate system is to use a more appropriate unit.
Lat/lon may be intuitive, but calculating distances with it can be tricky.
We can instead convert to UTM which uses meters as its units.
This makes it much easier to perform spatial operations, such as creating a 5km buffer around a specific point.

You will often work with different spatial objects that come in different coordinate reference systems.
Before integrating them, you must convert them into the same system.
If you don't, you will not be matching the same points on earth!

Let's map our RGB image to UTM 20

In [ ]:
from rasterio.warp import reproject, calculate_default_transform, Resampling

# This CRS corresponds to the best UTM region for our data
# For illustrative purposes, no need to dive deep here.
dst_crs = 'EPSG:32720'

projected_stack = []

for band in ["B04", "B03", "B02"]:
    # Get the path to the TIFF with the band info
    path = day1_channels[band]

    # Read the raster, take the first channel, and scale it
    with rasterio.open(path, "r") as src:

        # First, we calculate the new transform and dimensions
        new_transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds,
        )

        # We prepare an empty array to store the reprojected data
        destination = np.zeros((height, width), dtype=src.dtypes[0])

        # Now we perform the actual warping
        reproject(
            source=rasterio.band(src, 1),
            destination=destination,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=new_transform,
            dst_crs=dst_crs,
            resampling=Resampling.nearest
        )

        projected_stack.append(destination / max_intensity)

# Combine bands for plotting
projected_stack = np.stack(projected_stack, axis=-1)

# Define the display extent based on our new transform and shape
left, bottom  = new_transform * (0, height)
right, top = new_transform * (width, 0)
extent = [left, right, bottom, top]

plt.imshow(projected_stack * 10, extent=(left, right, bottom, top))
plt.title("Reprojected True Color (UTM 20S)")
plt.xlabel("Eastings (meters)")
plt.ylabel("Northings (meters)")

## Extracting Values

To extract specific data from our raster, we often want to look at a small area rather than the whole image.
For example, we might want to know the average reflectance around a specific study site.
Since we are now using a UTM coordinate system, we can easily define this area in meters.

First, we define a center point using UTM coordinates.
Then, we create a "buffer" around it.
In a meter-based system, a buffer of 5,000 creates a circle with a 5km radius.
We can use this to create a mask, which acts like a cookie cutter to isolate the pixels we are interested in.

In [ ]:
from rasterio.features import geometry_mask
from shapely.geometry import Point
from shapely.plotting import plot_polygon

# Define a center point in UTM coordinates
center_x, center_y = 200_000, 8_950_000
center_point = Point(center_x, center_y)

# Create a circle with a 5km radius
buffer_zone = center_point.buffer(5000)

# Create the mask
# 'invert=True' ensures that pixels INSIDE the circle are True
mask = geometry_mask(
    [buffer_zone],
    out_shape=projected_stack.shape[:2],
    transform=new_transform,
    invert=True
)

# Apply the mask to the Red channel
# We use the mask to filter our array before taking the mean
red_channel = projected_stack[:, :, 0]
mean_red_circular = red_channel[mask].mean()

print(f"Mean reflectance in the 1km buffer: {mean_red_circular:.4f}")

fig, ax = plt.subplots()
ax.imshow(projected_stack * 10, extent=(left, right, bottom, top))
plot_polygon(buffer_zone, ax=ax, add_points=False, color="red")
ax.set_title("Reprojected True Color (UTM 20S)")
ax.set_xlabel("Eastings (meters)")
ax.set_ylabel("Northings (meters)")

## Indices

Now that we can handle spatial data and extract values, we can look at how to analyse the landscape for ecological purposes.
One of the most common ways to do this is by creating Vegetation Indices.

The idea is to take two or more spectral bands and combine them into a single number for every pixel.
This is a classic example of hand-crafted features, using our knowledge of physics and biology to design a specific formula that highlights a certain property, like plant health or water content.
It is worth noting that while these indices are commonplace in traditional analysis, modern deep learning models usually work with the raw bands directly.

### The NDVI Formula

The most famous index is the **Normalized Difference Vegetation Index (NDVI)**.
It works because healthy green plants reflect a lot of Near-Infrared (NIR) light but absorb most of the Red light for photosynthesis.
By comparing these two, we can estimate where the vegetation is abundant and doing well.

The formula is:

$$ \text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}} $$

This formula scales everything between -1 and 1.
Generally, values close to 1 represent dense, healthy forests, while values near 0 might be bare soil or urban areas.
Water usually has negative values.

In [ ]:
# To calculate NDVI, we need NIR (B08) and Red (B04)

with rasterio.open(day1_channels["B08"]) as src:
    nir = src.read()[0] / max_intensity

with rasterio.open(day1_channels["B04"]) as src:
    red = src.read()[0] / max_intensity

# Calculate NDVI
# We add a tiny number (1e-10) to the denominator to avoid 'division by zero' errors
ndvi = (nir - red) / (nir + red + 1e-10)

# Plot the result
plt.imshow(ndvi, cmap="RdYlGn")
plt.colorbar(label="NDVI Value")
plt.title("Vegetation Health (NDVI)")
plt.axis("off")
plt.show()

#### 🖌️NVDI at 5km buffer

To wrap everything up, have a got at combining our of coordinate systems, buffers, and vegetation indices to calculate the average NDVI for a specific 5km area.